In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS f1.gold;

In [0]:
from pyspark.sql.functions import (col,avg, min,max,count)

In [0]:
pit_stops_df = spark.read.table(
    "f1.silver.pit_stops"
)

drivers_df = spark.read.table(
    "f1.silver.drivers"
)

constructors_df = spark.read.table(
    "f1.silver.constructors"
)

races_df = spark.read.table(
    "f1.silver.races"
)

results_df = spark.read.table(
    "f1.silver.results"
)

In [0]:
pit_stop_analysis_df = pit_stops_df \
    .join(
        drivers_df,
        pit_stops_df.driver_id == drivers_df.driver_id,
        "left"
    ) \
    .join(
        results_df.select(
            "race_id",
            "driver_id",
            "constructor_id"
        ),
        (
            (pit_stops_df.race_id == results_df.race_id) &
            (pit_stops_df.driver_id == results_df.driver_id)
        ),
        "left"
    ) \
    .join(
        constructors_df,
        results_df.constructor_id == constructors_df.constructor_id,
        "left"
    ) \
    .join(
        races_df,
        pit_stops_df.race_id == races_df.race_id,
        "left"
    )

In [0]:
pit_stop_analysis_df = pit_stop_analysis_df.filter(
    col("pit_stop_milliseconds").isNotNull()
)

In [0]:
from pyspark.sql.functions import when

pit_stop_analysis_final_df = pit_stop_analysis_df.groupBy(

    races_df.year,
    
    constructors_df.constructor_id,
    constructors_df.constructor_name,

    drivers_df.driver_id,
    drivers_df.name

).agg(

    count("*").alias("total_pit_stops"),

    min(
        when(col("pit_stop_milliseconds") != "\\N", col("pit_stop_milliseconds").cast("double"))
    ).alias("fastest_pit_stop_ms"),

    max(
        when(col("pit_stop_milliseconds") != "\\N", col("pit_stop_milliseconds").cast("double"))
    ).alias("slowest_pit_stop_ms"),

    avg(
        when(col("pit_stop_milliseconds") != "\\N", col("pit_stop_milliseconds").cast("double"))
    ).alias("avg_pit_stop_ms")
)

In [0]:
pit_stop_analysis_final_df = pit_stop_analysis_final_df.orderBy(
    col("avg_pit_stop_ms").asc()
)

In [0]:
pit_stop_analysis_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year") \
    .saveAsTable("f1.gold.pit_stop_analysis")

In [0]:
%sql
SELECT *
FROM f1.gold.pit_stop_analysis
LIMIT 20;